# Async Functions in Python

**Async** lets your program **do other work while waiting**, instead of sitting idle.

It is useful when your code has to **wait** for slow things — like a website to reply, a file to download, or a database to answer.

## Real-life analogy: the kitchen

Imagine you are **cooking rice and vegetables**:

- **One stove (synchronous):** you can cook only **one dish at a time**. Cook the rice first, wait until it's fully done, and **only then** start the vegetables.
- **Two stoves (async):** you put **rice on stove 1 and vegetables on stove 2** and cook **both at the same time**. You finish much faster.

Async is like having **two stoves** — you don't waste time waiting for one dish before starting the next.

## Sync vs Async

| | One stove (sync) | Two stoves (async) |
|---|------------------|--------------------|
| How many at once | One dish at a time | Many dishes together |
| While waiting | Does nothing | Cooks the other dish |
| Keyword to define | `def` | `async def` |
| Keyword to wait | (just call it) | `await` |
| Best for | Quick CPU work | Waiting on slow things (web, files) |

## Step 1: A simple async function

- Use **`async def`** to create one.
- Use **`await`** to wait for something slow without blocking.

> Note: In a normal `.py` file you start it with `asyncio.run(say_hello())`.
> In a **Jupyter notebook** there is already an event loop running, so we just write **`await`** directly.

In [1]:
import asyncio

async def say_hello():
    print("Hello...")
    await asyncio.sleep(1)   # wait 1 second, but don't block everything
    print("...World!")

# In a .py script:  asyncio.run(say_hello())
# In Jupyter we use await directly:
await say_hello()

Hello...
...World!


## Step 2: ONE stove (synchronous)

We cook with a single stove. Rice takes 3 seconds, vegetables take 2 seconds.

Because there is **only one stove**, we must finish the rice **before** we can start the vegetables.
Total time = 3 + 2 = **about 5 seconds**.

In [6]:
import time

def cook_one_stove(dish, seconds):
    print(f"Start cooking {dish}...")
    time.sleep(seconds)            # the stove is busy with this dish
    print(f"{dish} is ready!")

start = time.time()

cook_one_stove("rice", 3)         # stove busy for 3s
cook_one_stove("vegetables", 2)   # only now can we start the vegetables

print(f"\nTotal time: {round(time.time() - start, 1)}s  (one stove = 3 + 2 = 5s)")

Start cooking rice...
rice is ready!
Start cooking vegetables...
vegetables is ready!

Total time: 5.0s  (one stove = 3 + 2 = 5s)


## Step 3: TWO stoves (async)

Now we cook **rice and vegetables at the same time** using `asyncio.gather`.

Both dishes cook together, so the total time is just the **longest** dish (the rice, 3s) —
**not** the sum. Total time = **about 3 seconds**.

In [7]:
import asyncio, time

async def cook(dish, seconds):
    print(f"Start cooking {dish}...")
    await asyncio.sleep(seconds)   # this stove waits, but the OTHER stove keeps cooking
    print(f"{dish} is ready!")
    return dish

start = time.time()

# Put both dishes on the two stoves at the same time
await asyncio.gather(
    cook("rice", 3),         # stove 1
    cook("vegetables", 2),   # stove 2
)

print(f"\nTotal time: {round(time.time() - start, 1)}s  (two stoves = only ~3s!)")

Start cooking rice...
Start cooking vegetables...
vegetables is ready!
rice is ready!

Total time: 3.0s  (two stoves = only ~3s!)


## What just happened?

| | One stove (sync) | Two stoves (async) |
|---|------------------|--------------------|
| Rice (3s) + Vegetables (2s) | cooked one after another | cooked together |
| Total time | ~5 seconds | ~3 seconds |

Same food, **less time** — because async used the waiting time to cook the other dish.

## Step 5: Both in one program (see the difference)

This single program runs the **sync** way and the **async** way one after another, then prints both times so you can see the difference clearly in the output.

In [8]:
import asyncio, time

# ---------- SYNC: one stove (cook one after another) ----------
def cook_sync(dish, seconds):
    time.sleep(seconds)

start = time.time()
cook_sync("rice", 3)
cook_sync("vegetables", 2)
sync_time = round(time.time() - start, 1)

# ---------- ASYNC: two stoves (cook at the same time) ----------
async def cook_async(dish, seconds):
    await asyncio.sleep(seconds)

start = time.time()
await asyncio.gather(cook_async("rice", 3), cook_async("vegetables", 2))
async_time = round(time.time() - start, 1)

# ---------- Show the difference ----------
print(f"SYNC  (one stove) : {sync_time}s")
print(f"ASYNC (two stoves): {async_time}s")
print(f"Async saved {round(sync_time - async_time, 1)}s by cooking at the same time!")

SYNC  (one stove) : 5.0s
ASYNC (two stoves): 3.0s
Async saved 2.0s by cooking at the same time!


## Step 4: Getting results back

Tasks can **return values**, and `gather` gives you all of them in a list (in the same order).

In [9]:
import asyncio

async def cook_and_serve(dish, seconds):
    await asyncio.sleep(seconds)
    return f"{dish} served"

results = await asyncio.gather(
    cook_and_serve("rice", 3),
    cook_and_serve("vegetables", 2),
)
print(results)   # ['rice served', 'vegetables served']

['rice served', 'vegetables served']


## Key points to remember

- **One stove = synchronous:** one task at a time, you wait for each to finish.
- **Two stoves = async:** many tasks run **at the same time**, so waiting time is shared.
- Define an async function with **`async def`**, and wait for slow steps with **`await`**.
- **`asyncio.gather(...)`** runs many tasks together and returns their results.
- Async time = the **longest** task, not the **sum** of all tasks (5s → 3s in our kitchen).
- Great for **web requests, files, and APIs** — not for heavy math.
- In a `.py` file you start it with **`asyncio.run(main())`**; in **Jupyter** you just use `await` directly.
- Many AI tools (like `crew.kickoff_async()`) are async for this reason.